# 4. inference

Notebook loadfine-tuning after OpenVLA-UAV, inputdrone FPV image and instruction, action. 

**target: **
- loadfine-tuning after checkpoint
- imageinference: image + instruction → action
- inference and 
- action and true value for 
- inferencevelocitytest

## 4.1 environment

In [ ]:
import sys
import os
import json
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

PROJECT_ROOT = "/root/autodl-tmp/claude/UAV-Flow/OpenVLA-UAV"
sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = Path("/root/autodl-tmp/claude/data/uav_flow_subset")
CHECKPOINT_DIR = Path("/root/autodl-tmp/claude/runs")
BASE_MODEL = "/root/autodl-tmp/claude/models/models--openvla--openvla-7b/snapshots/47a0ec7fc4ec123775a391911046cf33cf9ed83f"

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

## 4.2 loadfine-tuning

In [ ]:
from prismatic.extern.hf.configuration_prismatic import OpenVLAConfig
from prismatic.extern.hf.modeling_prismatic import OpenVLAForActionPrediction
from prismatic.extern.hf.processing_prismatic import PrismaticProcessor, PrismaticImageProcessor
from transformers import AutoConfig, AutoImageProcessor, AutoProcessor, AutoModelForVision2Seq

AutoConfig.register("openvla", OpenVLAConfig)
AutoImageProcessor.register(OpenVLAConfig, PrismaticImageProcessor)
AutoProcessor.register(OpenVLAConfig, PrismaticProcessor)
AutoModelForVision2Seq.register(OpenVLAConfig, OpenVLAForActionPrediction)

# new checkpoint
checkpoints = [d for d in CHECKPOINT_DIR.iterdir() if d.is_dir() and (d / 'config.json').exists()]
if checkpoints:
model_path = str(max(checkpoints, key=lambda d: d.stat().st_mtime))
print(f"fine-tuning checkpoint: {model_path}")
else:
model_path = BASE_MODEL
print(f" checkpoint, use: {model_path}")

# load
print("\nload...")
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
model_path,
torch_dtype=torch.bfloat16,
low_cpu_mem_usage=True,
trust_remote_code=True,
).to("cuda:0")
model.eval()

# loadactionstatistics
stats_path = Path(model_path) / 'dataset_statistics.json'
if stats_path.exists():
with open(stats_path) as f:
model.norm_stats = json.load(f)
print(f"loadactionstatistics: {stats_path}")

print(f"\nloadcomplete!: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 4.3 imageinference

In [ ]:
from prismatic.vla.action_tokenizer import ActionTokenizer

action_tokenizer = ActionTokenizer(processor.tokenizer)

def predict_action(image, instruction, unnorm_key="sim"):
"""inputimage and instruction, action"""
prompt = f"In: What action should the robot take to {instruction.lower()}?\nOut:"
inputs = processor(prompt, image).to("cuda:0", dtype=torch.bfloat16)

with torch.no_grad():
# generateaction token
generated_ids = model.generate(
**inputs,
max_new_tokens=4, # 4 action
do_sample=False,
)

# decodingaction token
action_token_ids = generated_ids[0, -4:].cpu().numpy()
normalized_action = action_tokenizer.decode_token_ids_to_actions(action_token_ids)

# normalization (if has statistics)
if hasattr(model, 'norm_stats') and model.norm_stats and unnorm_key in model.norm_stats:
stats = model.norm_stats[unnorm_key]['action']
q01 = np.array(stats['q01'])
q99 = np.array(stats['q99'])
action = 0.5 * (normalized_action + 1) * (q99 - q01) + q01
else:
action = normalized_action

return action, normalized_action

In [ ]:
# test
test_traj = sorted(list(DATA_DIR.iterdir()))[10]
with open(test_traj / 'log.json') as f:
log = json.load(f)

# use in between frame
mid_idx = len(list(test_traj.glob('*.jpg'))) // 2
img_path = test_traj / f"{mid_idx:06d}.jpg"
image = Image.open(img_path).convert("RGB")
instruction = log['instruction']

# inference
action, norm_action = predict_action(image, instruction)

# display
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(image)
ax.set_title(f"Instruction: {instruction}", fontsize=11)
ax.axis('off')

# use represents toward (x= before after, y=)
cx, cy = 128, 128 # image in 
scale = 200
ax.annotate('', xy=(cx + action[1]*scale, cy - action[0]*scale),
xytext=(cx, cy),
arrowprops=dict(arrowstyle='->', color='red', lw=3))
ax.text(10, 245, f"Action: x={action[0]:.3f} y={action[1]:.3f} z={action[2]:.3f} yaw={action[3]:.3f}",
color='white', fontsize=10, bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
plt.show()

print(f"\naction (normalization): {action}")
print(f"action (normalization [-1,1]): {norm_action}")
print(f" x={action[0]:+.4f} ( before after) y={action[1]:+.4f} () z={action[2]:+.4f} ( above below) yaw={action[3]:+.4f} ()")

## 4.4 inference: vs true value

In [ ]:
# for frameinference
test_traj = sorted(list(DATA_DIR.iterdir()))[5]
with open(test_traj / 'log.json') as f:
log = json.load(f)

images = sorted(list(test_traj.glob('*.jpg')))
instruction = log['instruction']
gt_actions = np.array(log['preprocessed_logs'])[:, [0, 1, 2, 5]] # x, y, z, yaw

print(f": {log['id']}")
print(f"instruction: {instruction}")
print(f"frame: {len(images)}")

# frameinference
pred_actions = []
for i, img_path in enumerate(images):
image = Image.open(img_path).convert("RGB")
action, _ = predict_action(image, instruction)
pred_actions.append(action)
if i % 5 == 0:
print(f" Frame {i}/{len(images)}...")

pred_actions = np.array(pred_actions)
print(f"inferencecomplete!")

In [ ]:
# Visualization: vs true value
dim_names = ['X ( before after)', 'Y ()', 'Z ( above below)', 'Yaw ()']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, (ax, name) in enumerate(zip(axes, dim_names)):
t = range(len(gt_actions))
ax.plot(t, gt_actions[:, i], 'b-o', markersize=2, label='Ground Truth', alpha=0.7)
ax.plot(t, pred_actions[:, i], 'r--x', markersize=3, label='Predicted', alpha=0.7)
ax.set_xlabel('Frame')
ax.set_ylabel(name)
ax.set_title(name)
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(f'Action Prediction vs Ground Truth\nInstruction: {instruction}', fontsize=13)
plt.tight_layout()
plt.show()

## 4.5 many 

In [ ]:
# for many in between frameinference
np.random.seed(123)
sample_dirs = np.random.choice(sorted(list(DATA_DIR.iterdir())), size=6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for ax, traj_dir in zip(axes, sample_dirs):
with open(traj_dir / 'log.json') as f:
log = json.load(f)

imgs = sorted(list(traj_dir.glob('*.jpg')))
mid = len(imgs) // 2
image = Image.open(imgs[mid]).convert("RGB")
instruction = log['instruction']

action, _ = predict_action(image, instruction)

ax.imshow(image)
ax.set_title(f"{instruction[:35]}...", fontsize=9)

# represents toward 
cx, cy = 128, 128
scale = 300
ax.annotate('', xy=(cx + action[1]*scale, cy - action[0]*scale),
xytext=(cx, cy),
arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
ax.text(5, 248, f"x={action[0]:+.2f} y={action[1]:+.2f} z={action[2]:+.2f} yaw={action[3]:+.2f}",
color='white', fontsize=8, bbox=dict(boxstyle='round', facecolor='black', alpha=0.6))
ax.axis('off')

plt.suptitle('Multi-sample Inference Results (red arrow = predicted direction)', fontsize=14)
plt.tight_layout()
plt.show()

## 4.6 inferencevelocitytest

In [ ]:
# inferencevelocity benchmark
test_image = Image.open(sorted(list(DATA_DIR.iterdir()))[0] / '000005.jpg').convert('RGB')
test_instruction = "fly toward the tree"

# Warmup
for _ in range(3):
_ = predict_action(test_image, test_instruction)

# Benchmark
n_runs = 20
times = []
for _ in range(n_runs):
start = time.time()
_ = predict_action(test_image, test_instruction)
times.append(time.time() - start)

avg_time = np.mean(times)
std_time = np.std(times)
fps = 1.0 / avg_time

print(f"inferencevelocity (n={n_runs}):")
print(f": {avg_time*1000:.1f} ms (±{std_time*1000:.1f} ms)")
print(f" inference: {fps:.1f} Hz")
print(f"\ndroneneed to ~6 Hz, before {'' if fps >= 6 else ' not '} need ")

In [ ]:
# 
del model
torch.cuda.empty_cache()
print(".")

## small 

- inferenceprocess: image + instruction → Processor → model.generate → decodingaction token → normalization
- inferenceneed to `dataset_statistics.json` in q01/q99 normalization
- toward 
- inferencevelocity 5-8 Hz ( GPU), 

** below:** in Notebook 05 in, evaluation in test above can. 